# 01 — SimFin Historical Data Pipeline (v2.2)

**Concentric Value Model · Empirical Validation · DBA Research**

**Window: 2016-Q1 → 2025-Q4 (40 quarters × 200 firms = up to 8,000 firm-quarter observations).**

This window matches the SimFin **Start tier** (10 years of bulk data). It captures pre-COVID normal markets (2016-2019), the COVID shock (2020), the 2022-2023 inflation/hiking cycle, and the 2023-2025 recovery — a fuller spread of regimes than the original 2013-2023 design.

**What this notebook does, with all post-audit fixes baked in:**
1. Loads SimFin bulk data INCLUDING banks + insurance schemas (financials = 20% of universe)
2. Joins `SimFinId → ticker` properly in the data layer (no more live hot-fixes)
3. Builds the point-in-time panel with proper PE/PB/market_cap/beta/TTM-ROE
4. Forward returns validated against data-horizon availability
5. **A hard validation gate that STOPS the notebook if coverage is too low** — you will never again silently get an 81%-NaN panel
6. Saves all raw bulk CSVs to Drive immediately, so you can cancel SimFin after one pull and still re-run forever

## Step 1 — Mount Drive + clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/cvm-research'
os.makedirs(f'{WORK_DIR}/data', exist_ok=True)
os.makedirs(f'{WORK_DIR}/data/raw', exist_ok=True)   # raw SimFin CSV cache
os.makedirs(f'{WORK_DIR}/output', exist_ok=True)
print(f'Working dir: {WORK_DIR}')

In [ ]:
!pip install simfin pyarrow --quiet

import os, sys, subprocess
# NOTE: repo is CVM-Research (capitalised). Git is case-sensitive on the path.
REPO_PATH = 'FelixDLanger/CVM-Research'
CLONE_DIR = '/content/cvm-research'

# Self-healing clone (handles stale/half-broken state, never bare-cd's a missing dir)
if os.path.exists(CLONE_DIR) and not os.path.exists(f'{CLONE_DIR}/cvm_research/__init__.py'):
    subprocess.run(['rm','-rf',CLONE_DIR])
if not os.path.exists(f'{CLONE_DIR}/cvm_research/__init__.py'):
    r = subprocess.run(['git','clone',f'https://github.com/{REPO_PATH}.git',CLONE_DIR],
                       capture_output=True, text=True)
    print('clone rc:', r.returncode, r.stderr[-200:] if r.returncode else '')
if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)

import cvm_research
from cvm_research.pit_data import (
    quarter_ends, TickerIndex, ensure_ticker_column,
    build_pit_panel, attach_forward_returns, market_index_daily_returns,
)
print(f'✓ cvm_research v{cvm_research.__version__} loaded')

## Step 2 — Configure SimFin (paid Start tier)

Add your API key to Colab Secrets (key icon → `SIMFIN_API_KEY`). The Start tier gives 10 years of bulk data + prices, which covers our 2016-2025 window.

In [ ]:
from google.colab import userdata
SIMFIN_API_KEY = userdata.get('SIMFIN_API_KEY')

import simfin as sf
sf.set_api_key(SIMFIN_API_KEY)
sf.set_data_dir(f'{WORK_DIR}/data')
print('SimFin configured (Start tier: 10y bulk data).')

## Step 3 — Configuration constants

Everything tunable lives here. To change the study window, edit these two lines.

In [ ]:
START_YEAR = 2016
END_YEAR   = 2025

# Validation thresholds — the notebook STOPS if these aren't met (audit fix #5)
MIN_FUNDAMENTAL_COVERAGE = 0.50   # >=50% of rows must have real fundamentals
MIN_FORWARD_RETURN_COVERAGE = 0.40  # >=40% must have a forward return
REQUIRE_EARLY_YEAR_RETURNS = START_YEAR + 1  # forward returns must exist by this year

print(f'Study window: {START_YEAR}-Q1 to {END_YEAR}-Q4')
print(f'Validation gates: fundamentals>={MIN_FUNDAMENTAL_COVERAGE:.0%}, '
      f'forward_returns>={MIN_FORWARD_RETURN_COVERAGE:.0%}, '
      f'returns must reach back to {REQUIRE_EARLY_YEAR_RETURNS}')

## Step 4 — Download SimFin bulk data (standard + banks + insurance)

**Audit fix #2:** financial-sector firms use specialised schemas. We load all three and concatenate, so the 41 FIN tickers (~20% of the universe) get real data instead of silently falling to baseline.

In [ ]:
import pandas as pd

markets = ['us', 'de', 'uk', 'fr', 'jp', 'cn']

def safe(loader, **kw):
    try:    return loader(**kw)
    except Exception: return pd.DataFrame()

per_market = {}
for mkt in markets:
    comp = safe(sf.load_companies, market=mkt)
    if len(comp) == 0:
        print(f'  {mkt.upper()}: SKIP (no companies)'); continue

    inc = pd.concat([safe(sf.load_income,   variant='quarterly', market=mkt),
                     safe(sf.load_income_banks,     variant='quarterly', market=mkt),
                     safe(sf.load_income_insurance, variant='quarterly', market=mkt)], ignore_index=True)
    bal = pd.concat([safe(sf.load_balance,  variant='quarterly', market=mkt),
                     safe(sf.load_balance_banks,    variant='quarterly', market=mkt),
                     safe(sf.load_balance_insurance,variant='quarterly', market=mkt)], ignore_index=True)
    cf  = pd.concat([safe(sf.load_cashflow, variant='quarterly', market=mkt),
                     safe(sf.load_cashflow_banks,   variant='quarterly', market=mkt),
                     safe(sf.load_cashflow_insurance,variant='quarterly', market=mkt)], ignore_index=True)
    prc = safe(sf.load_shareprices, variant='daily', market=mkt)

    per_market[mkt] = {'companies':comp,'income':inc,'balance':bal,'cashflow':cf,'prices':prc}
    print(f'  {mkt.upper()}: {len(comp):>5} cos · {len(inc):>6} inc · {len(bal):>6} bal · {len(prc):>8} prices')

print('Download complete.')

## Step 5 — Save raw CSVs to Drive immediately

So you can **cancel SimFin after this one pull** and still re-run everything from the Drive cache forever.

In [ ]:
for mkt, d in per_market.items():
    for kind, df in d.items():
        if len(df):
            df.to_parquet(f'{WORK_DIR}/data/raw/{mkt}_{kind}.parquet')
print(f'Raw SimFin data cached to {WORK_DIR}/data/raw/')
print('You may cancel your SimFin subscription after this point — re-runs read from Drive.')

## Step 6 — Normalise columns + concatenate markets

In [ ]:
from cvm_research.pit_data import _normalise_columns

def norm(df): return _normalise_columns(df)

all_companies = pd.concat([norm(d['companies']) for d in per_market.values()], ignore_index=True)
all_income    = pd.concat([norm(d['income'])    for d in per_market.values()], ignore_index=True)
all_balance   = pd.concat([norm(d['balance'])   for d in per_market.values()], ignore_index=True)
all_cashflow  = pd.concat([norm(d['cashflow'])  for d in per_market.values()], ignore_index=True)
all_prices    = pd.concat([norm(d['prices'])    for d in per_market.values()], ignore_index=True)

print(f'companies: {len(all_companies):,}  income: {len(all_income):,}  '
      f'balance: {len(all_balance):,}  prices: {len(all_prices):,}')
print(f'companies columns: {list(all_companies.columns)[:12]}')

## Step 7 — Join SimFinId → ticker (audit fix: the live hot-fix, now in the data layer)

SimFin statements key on `simfin_id`, not ticker. `ensure_ticker_column` merges ticker from the companies table. This is the fix that previously had to be patched live each session.

In [ ]:
all_income   = ensure_ticker_column(all_income,   all_companies)
all_balance  = ensure_ticker_column(all_balance,  all_companies)
all_cashflow = ensure_ticker_column(all_cashflow, all_companies)
all_prices   = ensure_ticker_column(all_prices,   all_companies)

print('Ticker join complete. Rows dropped as unmapped:')
print(f'  income:   {all_income.attrs.get("dropped_unmapped", 0)}')
print(f'  balance:  {all_balance.attrs.get("dropped_unmapped", 0)}')
print(f'  cashflow: {all_cashflow.attrs.get("dropped_unmapped", 0)}')
print(f'  prices:   {all_prices.attrs.get("dropped_unmapped", 0)}')

## Step 8 — Build per-ticker indexes

In [ ]:
income_idx   = TickerIndex(all_income,   date_col='publish_date')
balance_idx  = TickerIndex(all_balance,  date_col='publish_date')
cashflow_idx = TickerIndex(all_cashflow, date_col='publish_date')
price_idx    = TickerIndex(all_prices,   date_col='date')
print(f'Indexes built. Tickers: inc={len(income_idx._per_ticker)} '
      f'bal={len(balance_idx._per_ticker)} px={len(price_idx._per_ticker)}')

## Step 9 — Reference universe + sector normalisation

In [ ]:
import urllib.request
TICKERS_URL = 'https://raw.githubusercontent.com/FelixDLanger/Concentric_Value_Model/main/tickers.csv'
tickers_path = f'{WORK_DIR}/tickers.csv'
urllib.request.urlretrieve(TICKERS_URL, tickers_path)

SECTOR_NORMALISE = {'HEALTH':'HC','IND':'INDUS','CONS_S':'CONS_D','REAL':'RE'}
ref = []
with open(tickers_path) as f:
    for line in f:
        if line.startswith('#') or line.startswith('ticker,'): continue
        p = line.strip().split(',')
        if len(p) >= 5:
            ref.append({'ticker':p[0],'name':p[1],'country':p[2],'region':p[3],
                        'sector':SECTOR_NORMALISE.get(p[4], p[4])})
ref_df = pd.DataFrame(ref)
print(f'Universe: {len(ref_df)} tickers')

# How many of our 200 are actually in the SimFin pull?
in_px = len(set(ref_df['ticker']) & set(all_prices['ticker']))
in_inc = len(set(ref_df['ticker']) & set(all_income['ticker']))
print(f'In price data: {in_px}/200   In income data: {in_inc}/200')

## Step 10 — Market index for beta (SPY w/ equal-weight fallback)

In [ ]:
mkt_returns = market_index_daily_returns(price_idx, index_ticker='SPY')
if len(mkt_returns) > 100:
    market_proxy = 'SPY'
    print(f'✓ SPY proxy: {len(mkt_returns):,} daily returns')
else:
    market_proxy = 'EW_UNIVERSE'
    print(f'⚠ SPY unavailable — building equal-weight universe proxy')
    by_date = {}
    for t in ref_df['ticker']:
        sub = price_idx._per_ticker.get(t)
        if sub is None or len(sub) < 2: continue
        cc = 'adj_close' if 'adj_close' in sub.columns else 'close'
        s = sub.set_index('date')[cc].dropna().sort_index().pct_change().dropna()
        for d, r in s.items(): by_date.setdefault(d, []).append(r)
    mkt_returns = pd.Series({d: sum(v)/len(v) for d,v in by_date.items() if len(v)>=50}).sort_index()
    print(f'  EW proxy: {len(mkt_returns):,} daily returns')
print(f'Market proxy: {market_proxy}')

## Step 11 — Build point-in-time panel

In [ ]:
QUARTERS = quarter_ends(START_YEAR, END_YEAR)
print(f'{len(QUARTERS)} quarter-ends: {QUARTERS[0].date()} → {QUARTERS[-1].date()}')

ticker_to_meta = {r['ticker']:{'sector':r['sector'],'country':r['country'],
                               'name':r['name'],'cap':None} for _,r in ref_df.iterrows()}

print('Building panel (3-5 min)...')
panel = build_pit_panel(
    tickers=ref_df['ticker'].tolist(), quarter_dates=QUARTERS,
    income_idx=income_idx, balance_idx=balance_idx, cashflow_idx=cashflow_idx,
    price_idx=price_idx, ticker_to_meta=ticker_to_meta, market_index_returns=mkt_returns)
print(f'Panel: {len(panel):,} rows')
for col in ['pe','pb','market_cap','beta','roe','debt_equity']:
    n = panel[col].notna().sum() if col in panel else 0
    print(f'  {col:12s}: {n:5d} ({100*n/len(panel):4.1f}%)')

## Step 12 — Forward returns (horizon-validated)

In [ ]:
panel = attach_forward_returns(panel, price_idx, trading_days=252)
n_fwd = panel['forward_return_pct'].notna().sum()
print(f'Forward returns: {n_fwd:,} / {len(panel):,} ({100*n_fwd/len(panel):.1f}%)')
panel['year'] = pd.to_datetime(panel['as_of']).dt.year
print('\nForward-return coverage by year:')
print(panel.groupby('year')['forward_return_pct'].apply(lambda s: s.notna().sum()).to_string())

## Step 13 — 🚦 VALIDATION GATE (audit fix #5)

**This cell STOPS the notebook with a loud error if the data is not good enough to analyse.** This is the single most important addition — it converts "silently broken" into "stops and tells you." If your paid pull was incomplete, you find out HERE, not after running a meaningless regression.

In [ ]:
errors = []

# Gate 1: fundamental coverage
fund_cols = [c for c in ['pe','pb','roe','debt_equity','market_cap'] if c in panel.columns]
has_fund = panel[fund_cols].notna().any(axis=1).mean()
if has_fund < MIN_FUNDAMENTAL_COVERAGE:
    errors.append(f'Fundamental coverage {has_fund:.1%} < required {MIN_FUNDAMENTAL_COVERAGE:.0%}. '
                  'SimFin pull likely incomplete — check tier/rate limits.')

# Gate 2: forward-return coverage
fwd_cov = panel['forward_return_pct'].notna().mean()
if fwd_cov < MIN_FORWARD_RETURN_COVERAGE:
    errors.append(f'Forward-return coverage {fwd_cov:.1%} < required {MIN_FORWARD_RETURN_COVERAGE:.0%}.')

# Gate 3: returns must reach back to early years (catches truncated price history)
early = panel[panel['year'] <= REQUIRE_EARLY_YEAR_RETURNS]['forward_return_pct'].notna().sum()
if early == 0:
    errors.append(f'ZERO forward returns at/before {REQUIRE_EARLY_YEAR_RETURNS}. '
                  'Price history did not load deep enough — this was the original bug.')

# Gate 4: all 4 quartiles should be reachable (composite spread sanity — checked after scoring in nb02)
# Gate 5: no duplicate observations
dup = panel.duplicated(subset=['ticker','as_of']).sum()
if dup > 0:
    errors.append(f'{dup} duplicate (ticker, as_of) rows — join fanned out.')

if errors:
    print('🛑 VALIDATION FAILED:\n')
    for e in errors: print(f'  ✗ {e}')
    raise RuntimeError('Panel failed validation — do NOT proceed to analysis. See errors above.')
else:
    print('✅ VALIDATION PASSED')
    print(f'  fundamental coverage: {has_fund:.1%}')
    print(f'  forward-return coverage: {fwd_cov:.1%}')
    print(f'  earliest-year returns present: {early} rows at/before {REQUIRE_EARLY_YEAR_RETURNS}')
    print(f'  duplicate rows: {dup}')

## Step 14 — Country macro (World Bank)

In [ ]:
import requests
WB = {'gdp_growth':'NY.GDP.MKTP.KD.ZG','inflation':'FP.CPI.TOTL.ZG'}
CMAP = {'US':'USA','DE':'DEU','NL':'NLD','FR':'FRA','DK':'DNK','CH':'CHE','IT':'ITA',
        'ES':'ESP','BE':'BEL','GB':'GBR','JP':'JPN','TW':'TWN','KR':'KOR','CN':'CHN',
        'HK':'HKG','IN':'IND','SG':'SGP'}
recs = []
for c2,c3 in CMAP.items():
    for lab,ind in WB.items():
        try:
            r = requests.get(f'https://api.worldbank.org/v2/country/{c3}/indicator/{ind}',
                             params={'format':'json','date':f'{START_YEAR}:{END_YEAR}','per_page':'30'}, timeout=15)
            d = r.json()
            if isinstance(d, list) and len(d)>1:
                for rec in d[1] or []:
                    if rec.get('value') is not None:
                        recs.append({'country':c2,'year':int(rec['date']),'indicator':lab,'value':float(rec['value'])})
        except Exception as e:
            print(f'  WB {c3}/{lab}: {e}')
macro_wide = pd.DataFrame(recs).pivot_table(index=['country','year'],columns='indicator',values='value').reset_index()
print(f'Macro: {len(macro_wide)} country-years')

In [ ]:
panel = panel.merge(macro_wide, on=['country','year'], how='left')
panel = panel.rename(columns={'gdp_growth':'macro_gdp','inflation':'macro_inflation'})
print(f'Panel with macro: {len(panel):,} rows')

## Step 15 — Save outputs

In [ ]:
panel.to_parquet(f'{WORK_DIR}/output/panel_pit_{START_YEAR}_{END_YEAR}.parquet')
macro_wide.to_parquet(f'{WORK_DIR}/output/macro_history.parquet')
panel.to_csv(f'{WORK_DIR}/output/panel_pit_{START_YEAR}_{END_YEAR}.csv', index=False)

import json
meta = {'build':'phase2-v2.3','window':f'{START_YEAR}-{END_YEAR}','n_quarters':len(QUARTERS),
        'n_obs':len(panel),'n_forward_returns':int(panel['forward_return_pct'].notna().sum()),
        'market_proxy':market_proxy,'fundamental_coverage':float(has_fund),
        'forward_return_coverage':float(fwd_cov)}
with open(f'{WORK_DIR}/output/panel_metadata.json','w') as f: json.dump(meta, f, indent=2, default=str)

# Standard filename the downstream notebooks expect
# Canonical filename matches the actual window. nb02/nb03 read this exact name.
print('Saved. Panel + macro + metadata in output/.')
print(f'Canonical file: panel_pit_{START_YEAR}_{END_YEAR}.parquet (nb02/nb03 read this exact name).')

## Summary

Clean panel, validated, reproducible:
- **2016-2025, 40 quarters, up to 8,000 observations**
- SimFinId→ticker join handled in the data layer (no hot-fixes)
- Banks + insurance schemas loaded (financials covered)
- Forward returns horizon-validated
- **Validation gate passed** — if you got here, the data is good enough to analyse
- Raw CSVs cached to Drive → cancel SimFin anytime

**Next:** `02_pca_analysis.ipynb` (Q-B), then `03_backtest_regression.ipynb` (Q-A). Both read the saved panel; no changes needed.